# Level 3 — Task 2: Support Vector Machine (SVM) for Classification
**Dataset:** Boston Housing | **Tools:** pandas, scikit-learn, matplotlib, seaborn

Target variable `MEDV` (median house value) is binarized at the median: **above median = 1 (High), below = 0 (Low)**.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

sns.set_theme(style="whitegrid")


## 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload house_Prediction_Data_Set.csv

cols = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT","MEDV"]
df = pd.read_csv("house_Prediction_Data_Set.csv", sep=r"\s+", header=None, names=cols)

print(f"Shape: {df.shape}")
df.head()


## 3. Exploratory Data Analysis

In [ ]:
df.describe()


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for ax, col in zip(axes.flatten(), df.columns):
    sns.histplot(df[col], kde=True, ax=ax, bins=25, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel("")

axes.flatten()[-1].set_visible(False)
plt.suptitle("Feature Distributions — Boston Housing", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.4)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 4. Create Binary Target & Preprocess

In [ ]:
# Binarize MEDV at median — High (1) vs Low (0) house price
median_price = df["MEDV"].median()
df["price_class"] = (df["MEDV"] > median_price).astype(int)

print(f"Median MEDV    : {median_price}")
print(f"Class balance  :")
print(df["price_class"].value_counts().rename({0: "Low (0)", 1: "High (1)"}))

X = df.drop(columns=["MEDV", "price_class"])
y = df["price_class"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")


## 5. Train SVM — Linear Kernel

In [ ]:
svm_linear = SVC(kernel="linear", probability=True, random_state=42)
svm_linear.fit(X_train, y_train)

y_pred_linear = svm_linear.predict(X_test)
y_prob_linear = svm_linear.predict_proba(X_test)[:, 1]

print("=== Linear Kernel ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_linear):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_linear):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_linear):.4f}")
print(f"F1        : {f1_score(y_test, y_pred_linear):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_prob_linear):.4f}")
print(f"CV Acc    : {cross_val_score(svm_linear, X_scaled, y, cv=5).mean():.4f}")


## 6. Train SVM — RBF Kernel

In [ ]:
svm_rbf = SVC(kernel="rbf", probability=True, random_state=42)
svm_rbf.fit(X_train, y_train)

y_pred_rbf = svm_rbf.predict(X_test)
y_prob_rbf = svm_rbf.predict_proba(X_test)[:, 1]

print("=== RBF Kernel ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_rbf):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_rbf):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_rbf):.4f}")
print(f"F1        : {f1_score(y_test, y_pred_rbf):.4f}")
print(f"AUC       : {roc_auc_score(y_test, y_prob_rbf):.4f}")
print(f"CV Acc    : {cross_val_score(svm_rbf, X_scaled, y, cv=5).mean():.4f}")


## 7. Kernel Comparison

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
linear_scores = [
    accuracy_score(y_test, y_pred_linear),
    precision_score(y_test, y_pred_linear),
    recall_score(y_test, y_pred_linear),
    f1_score(y_test, y_pred_linear),
    roc_auc_score(y_test, y_prob_linear),
]
rbf_scores = [
    accuracy_score(y_test, y_pred_rbf),
    precision_score(y_test, y_pred_rbf),
    recall_score(y_test, y_pred_rbf),
    f1_score(y_test, y_pred_rbf),
    roc_auc_score(y_test, y_prob_rbf),
]

comparison_df = pd.DataFrame({"Linear": linear_scores, "RBF": rbf_scores}, index=metrics)
print(comparison_df.round(4))

comparison_df.plot(kind="bar", figsize=(10, 5), edgecolor="white", color=["#4C72B0","#DD8452"])
plt.title("SVM Kernel Comparison — Linear vs RBF")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0.7, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


## 8. ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, y_prob in [("Linear", y_prob_linear), ("RBF", y_prob_rbf)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — SVM Kernels")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Decision Boundary (PCA 2D Projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_train_pca, X_test_pca, _, _ = train_test_split(X_pca, y, test_size=0.2, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, kernel, title in zip(axes, ["linear", "rbf"], ["Linear Kernel", "RBF Kernel"]):
    clf = SVC(kernel=kernel, probability=True, random_state=42)
    clf.fit(X_train_pca, y_train)

    h = 0.05
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")

    for cls, color, label in [(0, "#4C72B0", "Low"), (1, "#DD8452", "High")]:
        mask = y_test == cls
        ax.scatter(X_test_pca[mask.values, 0], X_test_pca[mask.values, 1],
                   s=30, color=color, label=label, edgecolors="white", linewidth=0.3)

    ax.set_title(f"Decision Boundary — {title}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend()

plt.suptitle("SVM Decision Boundaries (PCA 2D)", fontweight="bold")
plt.tight_layout()
plt.show()


## 10. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_linear, y_pred_rbf],
    ["Linear Kernel", "RBF Kernel"]
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Low", "High"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)

plt.suptitle("Confusion Matrices — SVM Kernels", fontweight="bold")
plt.tight_layout()
plt.show()


## 11. Summary

| Kernel | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| Linear | See output | See output | See output | See output | See output |
| RBF | See output | See output | See output | See output | See output |

RBF kernel generally outperforms linear on non-linearly separable data. AUC measures the model's ability to discriminate between classes across all thresholds — higher is better.